# 04 - Visualisations

Figures are written to `results/figures/`.

**Reported in the manuscript** (`results/figures/`):

- `fig1_societal_country.png` - Figure 1, Distribution of Societal Actor Categories by Country
- `fig2_political_country.png` - Figure 2, Distribution of Political Actor Categories and the
  Parliamentary Seat Share of the Governing Parties by Country

**Supplementary** (`results/figures/supplementary/`, prefix `figS_`): articles per day and per
week, weekly diversity overall and by topic, composition by topic and by outlet ownership,
public service media by campaign period, and party representation. These are not reported in
the manuscript and are kept for reference only.

The grey band in the time-series figures marks the campaign period (the final weeks before
each election).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

import helpers as h

h.set_apa_style()          # replaces the old rcParams block (that one turned grid ON)
FIG_DIR = h.RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
# Figures 1 and 2 go in FIG_DIR; everything else is supplementary and is not
# reported in the manuscript, so it is kept in a subfolder of its own.
(FIG_DIR / "supplementary").mkdir(parents=True, exist_ok=True)

soc = pd.read_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP)
soc["date"] = pd.to_datetime(soc["date"], format="mixed")
soc["topic_label"] = soc["topic"].map(h.TOPIC_LABEL).fillna(soc["topic"])
pol = pd.read_csv(h.POLITICAL_CLEAN, sep=h.CSV_SEP)
pol["topic_label"] = pol["topic"].map(h.TOPIC_LABEL).fillna(pol["topic"])

soc_cw  = pd.read_csv(h.MODEL_DIR / "soc_div_country_week.csv")
pol_cw  = pd.read_csv(h.MODEL_DIR / "pol_div_country_week.csv")
soc_cwt = pd.read_csv(h.MODEL_DIR / "soc_div_country_week_topic.csv")
pol_cwt = pd.read_csv(h.MODEL_DIR / "pol_div_country_week_topic.csv")
for d in (soc_cw, pol_cw, soc_cwt, pol_cwt):
    d["week_start"] = h.week_start(d["year_week"])

election = soc.groupby("country")["election_date"].first().map(pd.to_datetime)
# Campaign window from the data, not from election_date: Turkey's 2023 election went
# to a runoff on 28 May, two weeks after the first round, so its campaign runs nine
# weeks and ends AFTER election_date. Shading to election_date cropped those weeks.
camp_span = (soc[soc["campaign_period"] == 1]
               .groupby("country")["date"].agg(campaign_start="min", campaign_end="max"))
ELECTION_ROUNDS = {"TR": ["2023-05-14", "2023-05-28"]}

def mark_campaign(ax, cc):
    """Shade the campaign window and dash each election round."""
    lo = camp_span.loc[cc, "campaign_start"]
    hi = camp_span.loc[cc, "campaign_end"]
    ax.axvspan(lo, hi, color="grey", alpha=0.15, zorder=0)
    for d in ELECTION_ROUNDS.get(cc, [election[cc]]):
        ax.axvline(pd.to_datetime(d), color="black", ls="--", lw=0.8, zorder=1)

CAT_LABEL = {"nr_politicians": "Government and politicians", "nr_experts_professionals": "Experts and professionals",
             "nr_civil_society": "Civil society", "nr_general_public": "General public"}
TOPIC_DISPLAY = {"immigration": "Immigration", "rising prices": "Rising prices"}
OKABE_ITO = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73",
             "vermillion": "#D55E00", "purple": "#CC79A7", "skyblue": "#56B4E9",
             "yellow": "#F0E442", "black": "#000000"}
SOC_COLORS = {"Government and politicians": OKABE_ITO["blue"], "Experts and professionals": OKABE_ITO["orange"],
              "Civil society": OKABE_ITO["green"], "General public": OKABE_ITO["purple"]}
POL_COLORS = {"Government": OKABE_ITO["vermillion"], "Opposition": OKABE_ITO["skyblue"]}
TOPIC_COLORS = {"immigration": OKABE_ITO["vermillion"], "rising prices": OKABE_ITO["skyblue"]}
SERIES_COLORS = {"societal": OKABE_ITO["blue"], "political": OKABE_ITO["vermillion"]}

# Redundant encoding, per APA: differentiation must not rely on colour alone, so
# every categorical fill also carries a pattern and every line a marker. Keeps the
# figures readable in greyscale print and under colour-vision deficiency.
HATCH = ["", "//", "..", "xx"]
TOPIC_MARKER = {"immigration": "o", "rising prices": "s"}
PANEL = "ABCDE"

def pct_label(ax, x, y, value):
    """Percentage label on a filled segment, on a white plate so it stays legible
    over any hatch pattern."""
    ax.text(x, y, f"{100 * value:.0f}", ha="center", va="center", fontsize=8, color="black",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))

print("loaded:", len(soc), "articles |", len(pol), "political rows")


## Composition helpers

In [ ]:
def soc_composition(df, keys):
    g = df.groupby(keys)[h.CAT_COLS].sum()
    g = g.div(g.sum(axis=1), axis=0)
    g.columns = [CAT_LABEL[c] for c in g.columns]
    return g

def pol_composition(df, keys):
    d = df.assign(Government=np.where(df["government"] == 1, df["nr_actors"].fillna(1), 0),
                  Opposition=np.where(df["government"] == 0, df["nr_actors"].fillna(1), 0))
    g = d.groupby(keys)[["Government", "Opposition"]].sum()
    return g.div(g.sum(axis=1), axis=0)

def stacked_bar(ax, comp, colors, title, ylabel="Share of actors"):
    x = np.arange(len(comp)); bottom = np.zeros(len(comp))
    for col in comp.columns:
        ax.bar(x, comp[col].values, bottom=bottom, label=col, color=colors[col], width=0.7)
        bottom += comp[col].values
    ax.set_xticks(x); ax.set_xticklabels([h.COUNTRY_NAME[c] for c in comp.index], rotation=25, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(False)
print("helpers ready")

## Supplementary: Articles per day (immigration vs rising prices)


In [ ]:
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 1, figsize=(13, 16))
for ax, cc in zip(axes, h.COUNTRY_ORDER):
    sub = soc[soc["country"] == cc]
    daily = sub.groupby(["topic_label", sub["date"].dt.floor("D")]).size().rename("n").reset_index()
    for tp in ["immigration", "rising prices"]:
        s = daily[daily["topic_label"] == tp]
        ax.plot(s["date"], s["n"], color=TOPIC_COLORS[tp], lw=1.1, label=TOPIC_DISPLAY[tp])
    mark_campaign(ax, cc)
    ax.set_title(h.COUNTRY_NAME[cc]); ax.set_ylabel("articles/day")
    ax.set_xlabel("date"); ax.set_xlim(sub["date"].min(), sub["date"].max())
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=6.5)
    ax.grid(True, axis="y", alpha=0.3)
axes[0].legend(title="topic", loc="upper right", bbox_to_anchor=(1, 1.5), fontsize=11)
fig.tight_layout()
h.save_apa(fig, FIG_DIR / "supplementary/figS_articles_per_day.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(6.5, 7.0))
for ax, pn, cc in zip(axes, PANEL, h.COUNTRY_ORDER):
    sub = soc[soc["country"] == cc]
    wk = sub.groupby(["topic_label", "year_week"]).size().rename("n").reset_index()
    wk["week_start"] = h.week_start(wk["year_week"])
    for tp in ["immigration", "rising prices"]:
        s = wk[wk["topic_label"] == tp].sort_values("week_start")
        ax.plot(s["week_start"], s["n"], color=TOPIC_COLORS[tp], marker=TOPIC_MARKER[tp],
                ms=2.5, lw=1.2, label=TOPIC_DISPLAY[tp])
    mark_campaign(ax, cc)
    ax.set_title(f"{pn}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=10)
    ax.set_xlim(wk["week_start"].min(), wk["week_start"].max())
    ax.set_yticks(np.arange(0, 251, 50))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
fig.supylabel("Articles per week", fontsize=11)
hl, lb = axes[0].get_legend_handles_labels()
fig.legend(hl, lb, loc="lower center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0.02, 0.03, 1, 1])
h.save_apa(fig, FIG_DIR / "supplementary/figS_articles_per_week.png")
plt.show()


## Supplementary: Weekly diversity (societal and political)


In [ ]:
SERIES = {"Societal Diversity": (SERIES_COLORS["societal"], "o"),
          "Political Diversity": (SERIES_COLORS["political"], "s")}
fig, axes = plt.subplots(5, 1, figsize=(6.5, 6.8), sharey=True)
for ax, pn, cc in zip(axes, PANEL, h.COUNTRY_ORDER):
    s = soc_cw[soc_cw["country"] == cc].sort_values("week_start")
    p = pol_cw[pol_cw["country"] == cc].sort_values("week_start")
    for (lab, (col, mk)), d in zip(SERIES.items(), (s, p)):
        ax.plot(d["week_start"], d["simpson_Dz"], color=col, marker=mk, ms=2.5, lw=1.2, label=lab)
    mark_campaign(ax, cc)
    ax.set_title(f"{pn}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=10)
    ax.set_ylim(0, 1); ax.set_yticks([0, .25, .5, .75, 1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
fig.supylabel("Simpson's $D_z$", fontsize=11)
hl, lb = axes[0].get_legend_handles_labels()
fig.legend(hl, lb, loc="lower center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0.02, 0.03, 1, 1])
h.save_apa(fig, FIG_DIR / "supplementary/figS_weekly_diversity.png")
plt.show()


## Supplementary: Weekly diversity by topic


In [ ]:
def weekly_by_topic(cwt, fname):
    fig, axes = plt.subplots(5, 1, figsize=(6.5, 6.8), sharey=True)
    for ax, pn, cc in zip(axes, PANEL, h.COUNTRY_ORDER):
        sub = cwt[cwt["country"] == cc]
        for tp in ["immigration", "rising prices"]:
            s = sub[sub["topic_label"] == tp].sort_values("week_start")
            ax.plot(s["week_start"], s["simpson_Dz"], color=TOPIC_COLORS[tp],
                    marker=TOPIC_MARKER[tp], ms=2.5, lw=1.1, label=TOPIC_DISPLAY[tp])
        mark_campaign(ax, cc)
        ax.set_title(f"{pn}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=10)
        ax.set_ylim(0, 1); ax.set_yticks([0, .25, .5, .75, 1])
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    fig.supylabel("Simpson's $D_z$", fontsize=11)
    hl, lb = axes[0].get_legend_handles_labels()
    fig.legend(hl, lb, loc="upper right", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1.02))
    fig.tight_layout(rect=[0.02, 0.03, 1, 1])
    h.save_apa(fig, FIG_DIR / fname)
    plt.show()

weekly_by_topic(soc_cwt, "supplementary/figS_weekly_societal_by_topic.png")
weekly_by_topic(pol_cwt, "supplementary/figS_weekly_political_by_topic.png")


## Manuscript Figures 1 and 2: Actor-group composition per country


In [ ]:
def apa_stacked_single(comp, colors, fname, label_min=0.05, benchmark=None):
    """Single-panel APA stacked composition figure (countries on x).

    `benchmark` (optional): country-indexed Series (0..1) drawn as a dashed line
    over each bar, e.g. the governing parties' seat share. Government is the
    bottom segment, so a segment boundary above the line means government actors
    are over-represented relative to their seats (the reflective benchmark).
    """
    # 4.6in axes, not 5.2: bbox_inches="tight" grows the saved figure to include the
    # legend, so the axes must be narrower to keep the total inside the 6.5in margin.
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    x = np.arange(len(comp)); bottom = np.zeros(len(comp))
    for col, hatch in zip(comp.columns, HATCH):
        v = comp[col].to_numpy()
        ax.bar(x, v, bottom=bottom, label=col, color=colors[col], width=0.7,
               edgecolor="white", linewidth=0.5, hatch=hatch)
        for xi, (b, vi) in enumerate(zip(bottom, v)):
            if vi >= label_min:
                pct_label(ax, xi, b + vi / 2, vi)
        bottom += v
    if benchmark is not None:
        bm = benchmark.reindex(comp.index).to_numpy()
        ax.hlines(bm, x - 0.39, x + 0.39, color="black", linestyle=(0, (4, 2)),
                  linewidth=1.4, zorder=5)
        for xi, bv in zip(x, bm):
            ax.text(xi - 0.43, bv, f"{100 * bv:.0f}", ha="right", va="center",
                    fontsize=7, color="black", zorder=6,
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                              edgecolor="none", alpha=0.9))
    ax.set_xticks(x)
    ax.set_xticklabels([h.COUNTRY_NAME[c] for c in comp.index], rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.01, .25))
    ax.set_yticklabels([f"{int(t * 100)}" for t in np.arange(0, 1.01, .25)])
    ax.set_ylabel("Percentage of actors")
    handles, labels = ax.get_legend_handles_labels()
    if benchmark is not None:
        handles = handles + [Line2D([0], [0], color="black", linestyle=(0, (4, 2)), lw=1.4)]
        labels = labels + ["Government seat share"]
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
               frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    h.save_apa(fig, FIG_DIR / fname)
    plt.show()

refl = h.reflective_gov_share(pol)   # governing parties' seat share, per country

apa_stacked_single(soc_composition(soc, "country").reindex(h.COUNTRY_ORDER),
                   SOC_COLORS, "fig1_societal_country.png")
apa_stacked_single(pol_composition(pol, "country").reindex(h.COUNTRY_ORDER),
                   POL_COLORS, "fig2_political_country.png", benchmark=refl)


## Supplementary: Actor-group composition per country and topic


In [ ]:
def apa_stacked_panels(comp, colors, fname, level, labels, label_min=0.05, benchmark=None):
    """Two-panel APA stacked composition figure.

    `level` is the index level to split panels on; `labels` maps its values to display
    titles and fixes the panel order.
    """
    fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.6), sharey=True)
    for ax, pn, lv in zip(axes, "AB", labels):
        sub = comp.xs(lv, level=level).reindex(h.COUNTRY_ORDER)
        x = np.arange(len(sub)); bottom = np.zeros(len(sub))
        for col, hatch in zip(sub.columns, HATCH):
            v = sub[col].to_numpy()
            ax.bar(x, v, bottom=bottom, label=col, color=colors[col], width=0.72,
                   edgecolor="white", linewidth=0.5, hatch=hatch)
            for xi, (b, vi) in enumerate(zip(bottom, v)):
                if vi >= label_min:
                    pct_label(ax, xi, b + vi / 2, vi)
            bottom += v
        if benchmark is not None:
            bm = benchmark.reindex(sub.index).to_numpy()
            ax.hlines(bm, x - 0.40, x + 0.40, color="black", linestyle=(0, (4, 2)),
                      linewidth=1.4, zorder=5)
            # The benchmark is country-level, so it is identical in both panels;
            for xi, bv in zip(x, bm):
                ax.text(xi - 0.44, bv, f"{100 * bv:.0f}", ha="right", va="center",
                        fontsize=7, color="black", zorder=6,
                        bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                                    edgecolor="none", alpha=0.9))
        ax.set_xticks(x)
        ax.set_xticklabels([h.COUNTRY_NAME[c] for c in sub.index], rotation=45, ha="right", fontsize=9)
        ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.01, .25))
        ax.set_yticklabels([f"{int(t * 100)}" for t in np.arange(0, 1.01, .25)])
        ax.set_title(f"{pn}. {labels[lv]}", loc="left", fontsize=10)
    axes[0].set_ylabel("Percentage of actors")
    hl, lb = axes[0].get_legend_handles_labels()
    if benchmark is not None:
        hl = list(hl) + [Line2D([0], [0], color="black", linestyle=(0, (4, 2)), lw=1.4)]
        lb = list(lb) + ["Government seat share"]
    fig.legend(hl, lb, loc="lower center", ncol=len(lb), frameon=False, fontsize=8,
               bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    h.save_apa(fig, FIG_DIR / fname)
    plt.show()

soc_ct = soc_composition(soc, ["country", "topic_label"])
pol_ct = pol_composition(pol, ["country", "topic_label"])
refl = h.reflective_gov_share(pol)

apa_stacked_panels(soc_ct, SOC_COLORS, "supplementary/figS_societal_topic.png",
                   level="topic_label", labels=TOPIC_DISPLAY)
apa_stacked_panels(pol_ct, POL_COLORS, "supplementary/figS_political_topic.png",
                   level="topic_label", labels=TOPIC_DISPLAY, benchmark=refl)


## Supplementary: Actor-group composition, public service media vs newspapers


In [ ]:
soc_pn = pd.read_csv(h.MODEL_DIR / "soc_div_psb_news_paired.csv")
pol_pn = pd.read_csv(h.MODEL_DIR / "pol_div_psb_news_paired.csv")

In [ ]:
soc_pn.head()

In [ ]:
# Collapse the three newspaper types into one group, matching the comparison in 04.
# Composition needs the actor COUNTS, which only exist in the article-level data.
OUTLET_GROUP = {"Public broadcaster": "Public broadcaster", "Newspaper": "Newspapers"}
for d in (soc, pol):
    d["outlet_group"] = np.where(d["type"] == "Public broadcaster",
                                 "Public broadcaster", "Newspaper")

soc_og = soc_composition(soc, ["country", "outlet_group"])
pol_og = pol_composition(pol, ["country", "outlet_group"])

apa_stacked_panels(soc_og, SOC_COLORS, "supplementary/figS_societal_outlet.png",
                   level="outlet_group", labels=OUTLET_GROUP)
apa_stacked_panels(pol_og, POL_COLORS, "supplementary/figS_political_outlet.png",
                   level="outlet_group", labels=OUTLET_GROUP, benchmark=refl)


## Supplementary: Representation per party, per country


In [ ]:
# --- party representation (figures 6 and 7 share these) ----------------------
# Parties below 2% of both coverage and seats are dropped: with all 70 the figure
# ran to 23.6 inches and the tail carried no information. The retained set still
# covers 96.5% (TR) to 99.8% (BE) of political actors; say so in the figure note.
MIN_SHARE = 0.02

party = (pol.groupby(["country", "party_final"])
            .agg(actors=("nr_actors", "sum"), gov=("government", "first"),
                 seats=("seats", "first")).reset_index())
party["cov"] = party.groupby("country")["actors"].transform(lambda s: s / s.sum())
party = party[(party["cov"] >= MIN_SHARE) | (party["seats"] >= MIN_SHARE)].copy()
counts = party.groupby("country").size().reindex(h.COUNTRY_ORDER)

pt = (pol.groupby(["country", "topic_label", "party_final"])
         .agg(actors=("nr_actors", "sum")).reset_index())
pt["cov"] = pt.groupby(["country", "topic_label"])["actors"].transform(lambda s: s / s.sum())

LEG = [Patch(facecolor=POL_COLORS["Government"], label="Government"),
       Patch(facecolor=POL_COLORS["Opposition"], hatch="//", label="Opposition"),
       Line2D([0], [0], marker="D", color="w", markerfacecolor="none",
              markeredgecolor="black", markersize=6, label="Seat share")]

def party_bars(ax, d, cov, xmax):
    """One country panel: coverage bars coloured by government/opposition, with the
    seat-share benchmark as an open diamond. `cov` is passed separately from `d` so
    the same panel geometry serves both the overall and the per-issue figures."""
    y = np.arange(len(d))
    for yi, (g, v) in enumerate(zip(d["gov"], cov)):
        ax.barh(yi, v, color=POL_COLORS["Government"] if g == 1 else POL_COLORS["Opposition"],
                hatch="" if g == 1 else "//", height=0.7, edgecolor="white",
                linewidth=0.4, zorder=2)
    ax.scatter(d["seats"], y, facecolor="none", edgecolor="black", marker="D",
               s=16, linewidth=0.8, zorder=3)
    ax.set_yticks(y); ax.set_ylim(-0.7, len(d) - 0.3); ax.set_xlim(0, xmax)
    ax.tick_params(axis="x", labelsize=8)


### Supplementary: Public service media by campaign period


In [ ]:
def psm_campaign_interaction(pn, dim_label, fname):
    """PSM vs newspaper mean weekly Dz in routine vs campaign weeks, per country.

    Two contrasts in one figure: the vertical distance between the lines is the
    outlet-type gap, the slope of each line is the campaign change, and
    non-parallel lines mean the outlet gap itself shifts during the campaign.
    Descriptive: the campaign means rest on 7 weeks (Turkey 9), and no test in 03
    covers this 2x2 split, so the figure carries no significance markers.
    """
    m = (pn.groupby(["country", "group", "campaign_period"])["simpson_Dz"]
           .mean().unstack("campaign_period"))          # columns [0, 1] = routine, campaign
    STYLE = {"Public broadcaster": dict(ls="-", marker="o", label="PSM"),
             "Newspaper": dict(ls=(0, (4, 2)), marker="o", markerfacecolor="white",
                               label="Newspapers")}
    fig, axes = plt.subplots(1, 5, figsize=(6.5, 2.4), sharey=True)
    for ax, pnl, cc in zip(axes, PANEL, h.COUNTRY_ORDER):
        yP = m.loc[(cc, "Public broadcaster")].to_numpy()
        yN = m.loc[(cc, "Newspaper")].to_numpy()
        for grp, y in (("Public broadcaster", yP), ("Newspaper", yN)):
            ax.plot([0, 1], y, color="black", lw=1.2, ms=4.5, **STYLE[grp])
        # Value labels: at each x the higher point is labelled above, the lower
        # below, so the labels cannot collide where the lines meet (IR, NL).
        for xi in (0, 1):
            hi, lo = (yP[xi], yN[xi]) if yP[xi] >= yN[xi] else (yN[xi], yP[xi])
            ax.annotate(f"{hi:.2f}", (xi, hi), xytext=(0, 5), textcoords="offset points",
                        ha="center", fontsize=6.5, annotation_clip=False)
            ax.annotate(f"{lo:.2f}", (xi, lo), xytext=(0, -11), textcoords="offset points",
                        ha="center", fontsize=6.5, annotation_clip=False)
        ax.set_title(f"{pnl}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=9)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Routine", "Campaign"], fontsize=8)
        ax.set_xlim(-0.3, 1.3); ax.set_ylim(0, 1); ax.set_yticks([0, .25, .5, .75, 1])
    axes[0].set_ylabel(f"Simpson's $D_z$ ({dim_label})", fontsize=9)
    hl, lb = axes[0].get_legend_handles_labels()
    fig.legend(hl, lb, loc="lower center", ncol=2, frameon=False, fontsize=8,
               bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    h.save_apa(fig, FIG_DIR / fname)
    plt.show()


psm_campaign_interaction(soc_pn, "societal", "supplementary/figS_societal_psm_campaign.png")
psm_campaign_interaction(pol_pn, "political", "supplementary/figS_political_psm_campaign.png")


In [ ]:
soc_cells = pd.read_csv(h.MODEL_DIR / "soc_div_outlet_week_topic.csv")
pol_cells = pd.read_csv(h.MODEL_DIR / "pol_div_outlet_week_topic.csv")


def psm_campaign_issue(cells, dim_label, fname):
    """PSM vs newspaper mean Dz in routine vs campaign weeks, per country and issue.

    Same encoding as psm_campaign_interaction, with issue added as rows: the
    vertical gap is the outlet-type difference, each line's slope the campaign
    change. Means are over outlet-week-issue cells (the models' unit), so the
    newspaper line averages the three newspapers. Descriptive, no significance
    markers: each PSM campaign point rests on a single broadcaster over 4-9 weeks.
    """
    d = cells.dropna(subset=["simpson_Dz"]).copy()
    d["group"] = d["type"].where(d["type"] == "Public broadcaster", "Newspaper")
    m = (d.groupby(["topic_label", "country", "group", "campaign_period"])["simpson_Dz"]
           .mean().unstack("campaign_period"))          # columns [0, 1] = routine, campaign
    STYLE = {"Public broadcaster": dict(ls="-", marker="o", label="PSM"),
             "Newspaper": dict(ls=(0, (4, 2)), marker="o", markerfacecolor="white",
                               label="Newspapers")}
    topics = list(TOPIC_DISPLAY)
    fig, axes = plt.subplots(len(topics), 5, figsize=(6.5, 4.6), sharey=True)
    for r, tp in enumerate(topics):
        for c, (pnl, cc) in enumerate(zip(PANEL, h.COUNTRY_ORDER)):
            ax = axes[r, c]
            yP = m.loc[(tp, cc, "Public broadcaster")].to_numpy()
            yN = m.loc[(tp, cc, "Newspaper")].to_numpy()
            for grp, y in (("Public broadcaster", yP), ("Newspaper", yN)):
                ax.plot([0, 1], y, color="black", lw=1.2, ms=4.5, **STYLE[grp])
            # Higher point labelled above, lower below; near the floor a label below
            # would hit the axis, so it goes above instead
            for xi in (0, 1):
                hi, lo = (yP[xi], yN[xi]) if yP[xi] >= yN[xi] else (yN[xi], yP[xi])
                ax.annotate(f"{hi:.2f}", (xi, hi), xytext=(0, 5), textcoords="offset points",
                            ha="center", fontsize=6, annotation_clip=False)
                below = (0, -10) if lo > 0.1 else (0, 5 if hi - lo > 0.1 else 14)
                ax.annotate(f"{lo:.2f}", (xi, lo), xytext=below, textcoords="offset points",
                            ha="center", fontsize=6, annotation_clip=False)
            if r == 0:                                   # country titles on the top row only
                ax.set_title(f"{pnl}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=8)
            ax.set_xticks([0, 1])                        # period labels on the bottom row only
            ax.set_xticklabels(["Routine", "Campaign"] if r == len(topics) - 1 else [], fontsize=8)
            ax.set_xlim(-0.3, 1.3); ax.set_ylim(0, 1); ax.set_yticks([0, .25, .5, .75, 1])
            ax.tick_params(axis="both", labelsize=7)
        axes[r, 0].set_ylabel(TOPIC_DISPLAY[tp], fontsize=9)   # issue name labels each row
    fig.supylabel(f"Simpson's $D_z$ ({dim_label})", fontsize=9, x=0.0)
    hl, lb = axes[0, 0].get_legend_handles_labels()
    fig.legend(hl, lb, loc="lower center", ncol=2, frameon=False, fontsize=8,
               bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0.02, 0.05, 1, 1])
    h.save_apa(fig, FIG_DIR / fname)
    plt.show()


psm_campaign_issue(soc_cells, "societal", "supplementary/figS_societal_psm_campaign_issue.png")
psm_campaign_issue(pol_cells, "political", "supplementary/figS_political_psm_campaign_issue.png")


### Supplementary: Party representation figure


In [ ]:
# Shared value scale so bar lengths are comparable across countries. Computed from
# the same filtered `party` frame the panels draw from, so it can't drift.
XMAX = max(party["cov"].max(), party["seats"].max()) * 1.08

fig, axes = plt.subplots(5, 1, figsize=(6.5, 0.125 * counts.sum() + 1.8),
                         gridspec_kw={"height_ratios": counts.values})
for ax, pn, cc in zip(axes, PANEL, h.COUNTRY_ORDER):
    d = party[party["country"] == cc].sort_values("cov")
    party_bars(ax, d, d["cov"].to_numpy(), XMAX)
    ax.set_yticklabels(d["party_final"], fontsize=8)
    ax.set_title(f"{pn}. {h.COUNTRY_NAME[cc]}", loc="left", fontsize=10)
axes[-1].set_xlabel("Share of political-actor coverage")
fig.legend(handles=LEG, loc="lower center", ncol=3, frameon=False, fontsize=8,
           bbox_to_anchor=(0.5, -0.015))
fig.tight_layout(rect=[0, 0.035, 1, 1])
h.save_apa(fig, FIG_DIR / "supplementary/figS_party_representation.png")
plt.show()

## Supplementary: Representation per party, per country and topic


In [ ]:
# Shared value scale across all ten panels. Restricted to the retained party set so
# it matches what is actually drawn, and taken from pt because within-issue shares
# run higher than the overall shares used in Figure 6.
kept = set(zip(party["country"], party["party_final"]))
pt_kept = pt[[(c, p) in kept for c, p in zip(pt["country"], pt["party_final"])]]
XMAX_TOPIC = max(pt_kept["cov"].max(), party["seats"].max()) * 1.08

fig, axes = plt.subplots(5, 2, figsize=(6.5, 0.120 * counts.sum() + 2.0),
                         gridspec_kw={"height_ratios": counts.values})
for i, cc in enumerate(h.COUNTRY_ORDER):
    d = party[party["country"] == cc].sort_values("cov")
    parties = d["party_final"].tolist()
    sub = {tp: pt[(pt["country"] == cc) & (pt["topic_label"] == tp)]
                 .set_index("party_final")["cov"] for tp in TOPIC_DISPLAY}
    for j, tp in enumerate(["immigration", "rising prices"]):
        ax = axes[i, j]
        party_bars(ax, d, np.array([sub[tp].get(p, 0.0) for p in parties]), XMAX_TOPIC)
        ax.set_yticklabels(parties if j == 0 else [], fontsize=7.5)
        if j == 0: ax.set_ylabel(f"{PANEL[i]}. {h.COUNTRY_NAME[cc]}", fontsize=9)
        if i == 0: ax.set_title(TOPIC_DISPLAY[tp], fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("Share of coverage", fontsize=9)
fig.legend(handles=LEG, loc="lower center", ncol=3, frameon=False, fontsize=8,
           bbox_to_anchor=(0.5, -0.012))
fig.tight_layout(rect=[0, 0.03, 1, 1])
h.save_apa(fig, FIG_DIR / "supplementary/figS_party_representation_topic.png")
plt.show()
